# Bai tap
### Thiet ke Auto Trade, chay 1p chay 1 lan - moi khi chay thi se quet du lieu Forex/ Crypto/ Chung khoan theo Timeframe 1m
### Cho ra tin hieu Buy/ Sell: MA10 > MA20 thi Buy, nguoc lai Sell

Data lay tu TradeMT5 (Forex)

# Get Data

In [1]:
from symtable import Symbol
import sys
sys.path.append("../Common")
import TradeMT5
import MetaTrader5 as mt5	
import pandas as pd
import numpy as np
import talib
import redis
from datetime import datetime

def scan_market():

	symbol = "EURUSD.sml"
	timeframe = mt5.TIMEFRAME_M1 
	# ##############################################Step 1: Get Data###########################################################
	data = TradeMT5.TradeMT5.getData_MT5(symbol, timeframe, 1000)

	# ##############################################Step 2: Proces Data - Buy/ Sell############################################
	data["MA10"] = talib.SMA(data["Close"], timeperiod=10)
	data["MA20"] = talib.SMA(data["Close"], timeperiod=20)
	data["Buy_Signal"] = (data["MA10"] > data["MA20"])
	data["Sell_Signal"] = (data["MA10"] < data["MA20"])
	print(data)
	print("============================")
	# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
	# Nếu có tín hiệu thì đẩy qua Redis
	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=6) # 0-5: CK, 6-10: Forex, 11-15: Crypto
	# Tạo hash key
	hash_key = 'Bai tap tai lop 1'
	last_record = data.iloc[-1].copy()

	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
			last_record['Insertdate'] = datetime.now().isoformat()
		print(last_record)   
	else:
		print(last_record)   
		print('Không có tín hiệu!')
		
	

# Theo Schedule

In [2]:
# import schedule

# schedule.every(1).minute.at(":00").do(scan_market)

# while True:
# 	schedule.run_pending()

# Theo While True

In [3]:
from datetime import datetime, timedelta
import schedule
import time
import random

# Danh sách các phút cụ thể bạn muốn hàm được chạy
run_minutes = list(range(0, 60, 1)) # 0,1,2,3,4...59 : Cac phut chung ta can chay

# Thiết lập thời gian chạy cuối cùng để theo dõi
setattr(scan_market, 'last_run', None)

while True: # Luon luon kiem tra
	current_time = datetime.now() # Lay thoi gian hien tai 
	current_minute = current_time.minute # Lay phut hien
	current_second = current_time.second # Lay giay hien tai
	
	# Kiểm tra xem phút hiện tại có nằm trong danh sách các phút cần chạy hàm không
	if current_minute in run_minutes and current_second == 0: # 0 giay
		# Kiểm tra xem đã chạy hàm trong phút hiện tại hay chưa
		last_run = getattr(scan_market, 'last_run', None)
		if last_run is None or last_run != current_minute:
			scan_market() # Cho chay
			# Lưu lại phút cuối cùng mà hàm đã chạy
			setattr(scan_market, 'last_run', current_minute)

                        Open     High      Low    Close  Volume  spread  \
time                                                                      
2025-10-09 23:55:00  1.15639  1.15646  1.15636  1.15645      47       9   
2025-10-09 23:56:00  1.15643  1.15646  1.15639  1.15646      24       9   
2025-10-09 23:57:00  1.15646  1.15646  1.15638  1.15639      43       9   
2025-10-09 23:58:00  1.15639  1.15647  1.15636  1.15645      99      10   
2025-10-10 00:05:00  1.15610  1.15620  1.15609  1.15616       9      92   
...                      ...      ...      ...      ...     ...     ...   
2025-10-10 16:36:00  1.15678  1.15678  1.15658  1.15661     124       6   
2025-10-10 16:37:00  1.15662  1.15663  1.15626  1.15627     169       6   
2025-10-10 16:38:00  1.15627  1.15647  1.15627  1.15642     198       7   
2025-10-10 16:39:00  1.15642  1.15642  1.15632  1.15638     145       7   
2025-10-10 16:40:00  1.15638  1.15701  1.15638  1.15701     252       6   

                     rea

KeyError: 'time'